# ATINA — fotografias reais das soluções (Kaggle)

**Novo fluxo: pesquisa de fotografias reais no Wikimedia Commons.** Não gera cenas artificiais. Guarda até três candidatas por desafio, com origem, autor e licença. Todas ficam pendentes de revisão humana; não há upload para Supabase.

O CSV privado continua apenas no Input do Kaggle. Nunca publiques respostas, pistas ou tokens.

## 1. Preparar ambiente
Executa uma vez por sessão. A pesquisa de referências não precisa de GPU; a geração de imagens originais na secção 6 precisa de GPU (T4 ou superior recomendado).

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
repo = Path('/kaggle/working/ATINA')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git','clone','--depth','1','--branch','feature/ai-solution-images','https://github.com/lufime/ATINA.git',str(repo)], check=True)
work = repo / 'tools' / 'ai-solution-images'
subprocess.run([sys.executable,'-m','pip','install','-q','diffusers','transformers','accelerate','safetensors','cairosvg','requests','pillow'], check=True)
script = work / 'search_real_photos.py'
assert script.is_file(), f'Pesquisa não encontrada: {script}'
print('Preparação concluída. Pesquisa pronta. Para gerar imagens novas na secção 6, ativa GPU nas opções da sessão.')

## 2. Encontrar o CSV privado
Adiciona o CSV através de **Input → Add Input**. Não o tornes público.

In [ ]:
import pandas as pd
import json
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
print('CSV encontrados:', len(csv_files))
if not csv_files:
    raise FileNotFoundError('Adiciona o CSV privado em Input → Add Input.')
preferred = [p for p in csv_files if 'supabase' in p.name.lower() or 'challenge' in p.name.lower()]
CSV_PATH = str(preferred[0] if preferred else csv_files[0])
df = pd.read_csv(CSV_PATH, nrows=5)
required = {'challenge_id','category','answer','clues'}
if 'challenges' in df.columns and len(df.columns) == 1:
    raw = pd.read_csv(CSV_PATH, nrows=1).iloc[0]['challenges']
    records = json.loads(raw)
    if not isinstance(records, list) or not records:
        raise ValueError("A coluna 'challenges' não contém uma lista JSON de desafios.")
    df = pd.DataFrame(records)
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Faltam campos no CSV/JSON: {missing}.')
print('CSV selecionado:', Path(CSV_PATH).name)
print('Formato reconhecido e campos obrigatórios confirmados. (Respostas ocultas.)')

## 3. Segurança — sem upload nesta fase
As fotografias são descarregadas apenas para a sessão privada do Kaggle. Não é necessária nenhuma chave secreta. O envio para Supabase permanece desligado.

In [ ]:
UPLOAD_TO_SUPABASE = False
print('Upload Supabase ativo:', UPLOAD_TO_SUPABASE)
print('Nenhuma chave secreta é necessária para o teste local.')

## 4. Pesquisar fotografias reais — apenas 5 desafios
Esta célula procura até três fotografias candidatas por solução no Wikimedia Commons. Guarda a origem, autoria e licença. Os resultados são sugestões, não aprovações automáticas.

In [ ]:
import sys
import subprocess
from pathlib import Path
repo = Path('/kaggle/working/ATINA')
script = repo / 'tools' / 'ai-solution-images' / 'search_real_photos.py'
if not script.is_file():
    raise FileNotFoundError('Gerador não encontrado. Executa primeiro a célula 1.')
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
preferred = [p for p in csv_files if 'supabase' in p.name.lower() or 'challenge' in p.name.lower()]
if not csv_files:
    raise FileNotFoundError('CSV privado não encontrado em /kaggle/input.')
CSV_PATH = str(preferred[0] if preferred else csv_files[0])
OUTPUT_DIR = '/kaggle/working/ATINA-real-photos'
cmd = [sys.executable, '-u', str(script), '--manifest', CSV_PATH, '--output', OUTPUT_DIR, '--limit', '5']
print('A pesquisar fotografias reais no Wikimedia Commons...', flush=True)
result = subprocess.run(cmd, cwd=str(script.parent), check=False)
generated = sorted(Path(OUTPUT_DIR).glob('*.webp'))
print('\nCódigo de saída:', result.returncode)
print('Fotografias candidatas encontradas:', len(generated))
for p in generated: print(p.name, round(p.stat().st_size/1024), 'KB')

## 5. Rever fotografias e respetivas fontes
Cada desafio pode ter até três fotografias candidatas. Confirma se a imagem representa realmente a solução e consulta o ficheiro JSON correspondente para verificar autor, licença e origem. Não publiques capturas com respostas privadas.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
# Voltar a localizar o CSV e as imagens, mesmo após reinício do Kernel.
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
preferred = [p for p in csv_files if 'supabase' in p.name.lower() or 'challenge' in p.name.lower()]
if not csv_files:
    raise FileNotFoundError('CSV privado não encontrado em /kaggle/input.')
CSV_PATH = str(preferred[0] if preferred else csv_files[0])
OUTPUT_DIR = Path('/kaggle/working/ATINA-real-photos')
raw_df = pd.read_csv(CSV_PATH)
if 'challenges' in raw_df.columns and len(raw_df.columns) == 1:
    records = json.loads(raw_df.iloc[0]['challenges'])
    review_df = pd.DataFrame(records)
else:
    review_df = raw_df
if not {'challenge_id','answer'}.issubset(review_df.columns):
    raise ValueError('O CSV não contém challenge_id e answer para fazer a revisão.')
generated = sorted(OUTPUT_DIR.glob('*.webp')) if OUTPUT_DIR.exists() else []
print('Imagens encontradas:', len(generated))
if not generated:
    print('Não há imagens nesta sessão. Executa primeiro a célula 4.')
for p in generated:
    challenge_id = int(p.name.split('-')[0].split('.')[0])
    match = review_df[review_df['challenge_id'].astype(str).str.strip() == str(challenge_id)]
    if match.empty:
        match = review_df[pd.to_numeric(review_df['challenge_id'], errors='coerce') == challenge_id]
    answer = str(match.iloc[0]['answer']) if not match.empty else 'Resposta não encontrada no CSV'
    print(f'Desafio {challenge_id}: {answer} — candidata {p.name}')
    with Image.open(p) as img:
        plt.figure(figsize=(4,4))
        plt.imshow(img)
        plt.title(p.name)
        plt.axis('off')
        plt.show()

## 6. Gerar fotografias originais com IA — apenas 5 desafios
Esta etapa é OBRIGATÓRIA para criar imagens novas. Usa Stable Diffusion 1.5 img2img com as fotografias reais candidatas da secção 4 como referência. Precisa de GPU ativa. Gera em /kaggle/working/ATINA-ai-generated. Não reutiliza nem mostra as fotografias de referência como resultado final. Não faz upload para Supabase.

**Importante:** a secção 5 mostra fotografias de referência; só a execução da célula seguinte (secção 6) cria ficheiros novos com IA.

In [ ]:
import sys, subprocess
from pathlib import Path
repo = Path('/kaggle/working/ATINA')
script = repo / 'tools' / 'ai-solution-images' / 'generate_from_reference.py'
import torch
if not torch.cuda.is_available(): raise RuntimeError('Ativa GPU em Session options → Accelerator → GPU antes de executar esta célula.')
output_dir = Path('/kaggle/working/ATINA-ai-generated')
if output_dir.exists():
    import shutil
    shutil.rmtree(output_dir)
csv_files = list(Path('/kaggle/input').rglob('*.csv'))
preferred = [p for p in csv_files if 'supabase' in p.name.lower() or 'challenge' in p.name.lower()]
if not csv_files: raise FileNotFoundError('CSV privado não encontrado.')
CSV_PATH = str(preferred[0] if preferred else csv_files[0])
cmd = [sys.executable,'-u',str(script),'--manifest',CSV_PATH,'--references','/kaggle/working/ATINA-real-photos','--output',str(output_dir),'--limit','5','--candidate-rank','1','--strength','0.28']
print('A gerar 5 imagens novas a partir das fotografias de referência. Confirma que GPU está ativa.',flush=True)
result = subprocess.run(cmd,cwd=str(script.parent),check=False)
print('Código de saída:',result.returncode)
print('FOTOGRAFIAS NOVAS GERADAS:',len(list(output_dir.glob('*.webp'))))

## 6.1 Rever as imagens novas criadas pela IA
Esta galeria lê exclusivamente ATINA-ai-generated. Se aparecer zero imagens, a geração ainda não foi executada com sucesso.

In [ ]:
from pathlib import Path
from PIL import Image
import json, matplotlib.pyplot as plt
out = Path('/kaggle/working/ATINA-ai-generated')
files = sorted(p for p in out.glob('*.webp') if '-candidate-' not in p.name) if out.exists() else []
print('IMAGENS ORIGINAIS CRIADAS PELA IA:', len(files))
if not files: print('Ainda não foram criadas imagens. Executa a célula da secção 6.')
for p in files:
    print(p.name)
    with Image.open(p) as im:
        plt.figure(figsize=(4,4)); plt.imshow(im); plt.axis('off'); plt.show()
    meta=p.with_name(p.stem+'-provenance.json')
    if meta.exists():
        d=json.loads(meta.read_text(encoding='utf-8'))
        print('Resposta:',d.get('answer'),'| Referência:',d.get('reference_file'),'| Estado:',d.get('review_status'))


## 7. Lote completo — NÃO executar ainda
Só depois de aprovar visualmente as cinco imagens originais criadas na secção 6. Esta etapa continuará desativada. Nenhum upload para Supabase é feito automaticamente.